# Topic 02 — Cardiovascular Visual Analysis

Reproduction of the public mlcourse.ai demo assignment on the original cardiovascular dataset. The dataset contains 70,000 medical examination records.

The focus here is not only obtaining an answer, but preserving a defensible analytical path: **assumption → calculation → visualization → check → conclusion**. Medical associations in this notebook are descriptive and must not be read as clinical advice or causal evidence.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
DATA_URL = 'https://raw.githubusercontent.com/Yorko/mlcourse.ai/main/data/'
df = pd.read_csv(DATA_URL + 'mlbootcamp5_train.csv', sep=';')
print('shape:', df.shape)
df.head()

## 1. Understand the encoding before interpreting it

The `gender` values are codes, not labels. I should not assume what `1` or `2` means. The assignment suggests using average height as a plausibility clue because men are taller on average at population level.

In [ ]:
gender_height = df.groupby('gender')['height'].agg(['count','mean','median'])
gender_height

In [ ]:
male_code = gender_height['mean'].idxmax()
female_code = gender_height['mean'].idxmin()
print('inferred male code:', male_code)
print('inferred female code:', female_code)
print('counts:', df['gender'].value_counts().to_dict())

**Reasoning.** This is an inference from an aggregate pattern, not a property encoded in the raw column name. I make the assumption explicit before using gender in later comparisons.

## 2. Alcohol and smoking by inferred gender

For binary behaviors, proportions by group answer the question more directly than raw counts because the gender groups are different sizes.

In [ ]:
behavior = df.groupby('gender')[['alco','smoke']].mean().rename(columns={'alco':'alcohol_share','smoke':'smoker_share'})
behavior

In [ ]:
smoking_gap_pp = 100 * abs(behavior.loc[male_code, 'smoker_share'] - behavior.loc[female_code, 'smoker_share'])
print('smoking gap, percentage points:', round(smoking_gap_pp, 1))

## 3. Age: inspect units before comparing groups

The raw age values are far too large to be years. The data description says age is stored in days, so I convert to years/months explicitly instead of interpreting the raw number.

In [ ]:
df = df.assign(age_years=df['age'] / 365.25, age_months=df['age'] / 30.4375)
age_by_smoke = df.groupby('smoke')['age_months'].median()
print(age_by_smoke)
print('median difference, months:', round(abs(age_by_smoke.diff().dropna().iloc[0])))

## 4. BMI as a derived feature

**Question.** Does body-mass distribution differ by gender and cardiovascular status?

I derive BMI from weight and height. A derived feature is useful when it expresses a relation that is hard to see from two raw columns independently.

In [ ]:
df['bmi'] = df['weight'] / (df['height'] / 100) ** 2
print('median BMI:', df['bmi'].median())
print(df.groupby('gender')['bmi'].mean())
print(df.groupby('cardio')['bmi'].median())

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=df, x='cardio', y='bmi', showfliers=False, ax=ax)
ax.set_ylim(10, 55)
ax.set_title('BMI distribution by cardiovascular status')
plt.show()

**Caution.** BMI is a rough population-level indicator. The plot can show association with `cardio`, but it cannot establish a causal medical relationship.

## 5. Clean implausible measurements

The assignment defines a limited cleaning rule: remove records with diastolic pressure above systolic pressure and values of height/weight outside the 2.5–97.5 percentile interval. I apply exactly that rule so the result is reproducible and comparable with the course assignment.

In [ ]:
h_low, h_high = df['height'].quantile([0.025, 0.975])
w_low, w_high = df['weight'].quantile([0.025, 0.975])
mask = (
    (df['ap_lo'] <= df['ap_hi'])
    & df['height'].between(h_low, h_high)
    & df['weight'].between(w_low, w_high)
)
clean = df.loc[mask].copy()
removed_pct = 100 * (1 - len(clean) / len(df))
print('rows before:', len(df))
print('rows after:', len(clean))
print('removed %:', round(removed_pct, 2))

**Reasoning.** I compare before/after counts because cleaning is itself an analytical decision. A rule that silently removes a large share of data deserves scrutiny.

## 6. Pearson correlation heatmap

**Question.** Which approximately linear relationships are strongest after cleaning?

A correlation heatmap is a screening view, not proof that one variable drives another.

In [ ]:
pearson = clean.select_dtypes(include='number').corr(method='pearson')
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(pearson, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Pearson correlations after cleaning')
plt.tight_layout()
plt.show()

gender_corr = pearson['gender'].drop('gender').abs().sort_values(ascending=False)
gender_corr.head()

## 7. Spearman rank correlation

Pearson emphasizes linear relationships. Spearman asks whether higher ranks of one feature tend to correspond to higher/lower ranks of another, which can reveal monotonic relationships even when they are not perfectly linear.

In [ ]:
spearman = clean.select_dtypes(include='number').corr(method='spearman')
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(spearman, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Spearman rank correlations')
plt.tight_layout()
plt.show()

pairs = spearman.where(np.triu(np.ones(spearman.shape), k=1).astype(bool)).stack().abs().sort_values(ascending=False)
pairs.head(10)

## 8. Height distribution and gender encoding

A violin plot makes the distribution-level difference visible and lets me check whether the earlier mean-height inference was supported by the full distributions rather than only one summary statistic.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.violinplot(data=clean, x='gender', y='height', inner='quartile', ax=ax)
ax.set_title('Height distribution by gender code')
plt.show()

## 9. Age and cardiovascular status

**Question.** At what ages does the observed class balance shift toward more people with CVD?

Age is converted to rounded years first. Then I compare counts inside each age value.

In [ ]:
clean['age_years_round'] = (clean['age'] / 365.25).round().astype(int)
age_counts = clean.groupby(['age_years_round','cardio']).size().unstack(fill_value=0)
age_counts.columns = ['no_cvd','cvd']
age_counts['cvd_minus_no_cvd'] = age_counts['cvd'] - age_counts['no_cvd']
print(age_counts.head())
ages_cvd_majority = age_counts.index[age_counts['cvd'] > age_counts['no_cvd']]
print('first age with CVD majority:', ages_cvd_majority.min() if len(ages_cvd_majority) else None)

plot_data = clean[clean['age_years_round'].between(40, 65)]
fig, ax = plt.subplots(figsize=(14, 5))
sns.countplot(data=plot_data, x='age_years_round', hue='cardio', ax=ax)
ax.set_title('Cardiovascular status by rounded age')
plt.show()

## 10. What I learned from the assignment

1. Encoding must be understood before categories are interpreted.
2. Proportions are often safer than raw counts when group sizes differ.
3. Derived features such as BMI can expose structure that is hidden in raw columns.
4. Cleaning changes the population being analyzed and must be quantified.
5. Pearson and Spearman answer related but different questions.
6. Correlation and visual separation are descriptive evidence, not causality.
7. A chart should lead either to a numerical check or to a new testable hypothesis.